# Entraînement des politiques A et D sur Kaggle

**Réglages du notebook (panneau de droite) :** Accelerator = **GPU T4 x2**, Internet = **On**.

Pour un entraînement long, ne pas garder l'onglet ouvert : **Save Version → Save & Run All (Commit)**.
Kaggle exécute alors tout en arrière-plan (jusqu'à 12 h) et garde `/kaggle/working` dans l'onglet *Output*.

- **A** : conditions normales, sans randomisation ni critique privilégié.
- **D** : chaque chien simulé tire sa friction, sa charge et une patte affaiblie (`scenario=tous`),
  et le critique voit ces paramètres.

Ensuite : évaluation des deux sur les grilles glace et patte, figures de comparaison,
vidéos côte à côte, et un zip à télécharger.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

## Installation
Le code est cloné depuis GitHub : il faut avoir poussé ses derniers commits.

In [ ]:
# Sortir du dossier avant de le supprimer : sinon, en relançant la cellule,
# le shell se retrouve dans un dossier effacé et tout échoue.
%cd /kaggle/working
!rm -rf /tmp/rescue_dog
!git clone --depth 1 https://github.com/BOUBAKERAMDY/rlRobot3D.git /tmp/rescue_dog
%cd /tmp/rescue_dog
!git log --oneline -1
!pip install -q -e . "jax[cuda12]<0.10"

In [ ]:
import os

# Rendu sans écran pour les vidéos, et option XLA recommandée par MuJoCo Playground.
os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"

!python -c "import jax; print(jax.devices())"

## Réglages
`QUICK = True` lance un essai court (10 M pas par politique, ~quelques minutes) pour mesurer la
vitesse du GPU avant de lancer les vrais entraînements. D passe en premier : si la session
s'arrête avant la fin, on a au moins la politique la plus importante.

In [ ]:
QUICK = False
STEPS = "10_000_000" if QUICK else "200_000_000"
RESULTS = "/kaggle/working/results"

POLICIES = {
    "D": "domain_randomization=true privileged_critic=true",
    "A": "domain_randomization=false privileged_critic=false",
}

## Entraînement
Chaque ligne `[x min] pas … récompense …` donne la vitesse réelle : pas ÷ minutes.

In [ ]:
for name, switches in POLICIES.items():
    print(f"===== Politique {name} =====")
    !python scripts/train.py scenario=tous run_name={name} output_dir={RESULTS} num_timesteps={STEPS} {switches}

## Courbes d'apprentissage

In [ ]:
import json
import matplotlib.pyplot as plt

for name in POLICIES:
    rows = [json.loads(line) for line in open(f"{RESULTS}/{name}/metrics.jsonl")]
    steps = [r["step"] for r in rows]
    reward = [r["eval/episode_reward"] for r in rows]
    plt.plot(steps, reward, marker="o", label=name)
# Récompense d'un chien qui reste debout sans bouger (mesurée sur 32 épisodes) :
# tant que la courbe reste près de cette ligne, la politique ne marche pas encore.
plt.axhline(17.7, color="gray", linestyle="--", label="debout immobile")
plt.legend()
plt.xlabel("pas d'environnement")
plt.ylabel("récompense d'évaluation")
plt.savefig(f"{RESULTS}/reward_A_D.png", dpi=120)
plt.show()

## Évaluation et comparaison
Politique figée, testée sur la grille glace (friction × charge) et la grille patte (patte × force).
Métrique principale : mission réussie = debout 20 s **et** au moins la moitié de la distance demandée.

In [ ]:
for name in POLICIES:
    !python scripts/evaluate.py {RESULTS}/{name}

for scenario in ("glace", "patte"):
    !python scripts/compare.py {scenario} A={RESULTS}/A D={RESULTS}/D --out {RESULTS}/comparaison

In [ ]:
from IPython.display import Image, display

for scenario in ("glace", "patte"):
    display(Image(f"{RESULTS}/comparaison/heatmaps_{scenario}.png"))
    display(Image(f"{RESULTS}/comparaison/classement_{scenario}.png"))

## Vidéos A contre D dans les conditions les plus dures

In [ ]:
from IPython.display import Video

!python scripts/duel_video.py A={RESULTS}/A D={RESULTS}/D --test glace friction=0.1 payload=5 --out {RESULTS}/comparaison/duel_glace.mp4
!python scripts/duel_video.py A={RESULTS}/A D={RESULTS}/D --test patte leg=FL strength=0.2 --out {RESULTS}/comparaison/duel_patte.mp4
Video(f"{RESULTS}/comparaison/duel_glace.mp4", embed=True, width=960)

## Téléchargement
Le zip apparaît dans l'onglet *Output* ; le décompresser dans `results/` sur le Mac.

In [ ]:
!cd {RESULTS} && zip -qr /kaggle/working/resultats_A_D.zip .
!ls -lh /kaggle/working/resultats_A_D.zip